# 1. Titulo y proposito

**Gradix** es un proyecto de pre-grading visual para cartas Pokemon TCG a partir de imagenes propias.

El objetivo tecnico del proyecto es detectar la carta dentro de una fotografia, rectificar su perspectiva, validar la calidad del warp y, despues, extraer features y scores utiles para un analisis preliminar de condicion.

Este notebook documenta de forma reproducible la obtencion del dataset tabular de Gradix sin usar Streamlit como interfaz. La app principal (`app.py`) se mantiene como componente de UI, mientras que aqui se reutiliza directamente la logica real del pipeline del proyecto.

Las imagenes de entrada provienen de capturas propias organizadas dentro de `data/raw/`.

# 2. Configuracion del entorno

La siguiente celda define la raiz del repo, ajusta `sys.path` si hace falta, fija las rutas de `data/raw` y `data/processed`, e importa las funciones reales que usa el pipeline activo.

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import pprint
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.axisbelow'] = True

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'generar_dataset.py').exists() and (candidate / 'src' / 'pipeline' / 'card_analysis.py').exists():
            return candidate
    raise FileNotFoundError(
        'No se encontro la raiz del repo. Ejecuta el notebook desde el repo Gradix o ajusta la ruta manualmente.'
    )

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
OUTPUT_BASE = PROCESSED_DIR / 'dataset_gradix'
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

from generar_dataset import (
    FEATURE_SCHEMA_VERSION,
    aplanar_diccionario,
    evaluar_estado_analisis,
    normalizar_label_condition,
    obtener_label_desde_raw,
    procesar_lote_imagenes,
    target_desde_label,
)
from src.pipeline.card_analysis import analyze_card_image

repo_status = pd.Series(
    {
        'ROOT': str(ROOT),
        'generar_dataset.py existe': (ROOT / 'generar_dataset.py').exists(),
        'src/pipeline/card_analysis.py existe': (ROOT / 'src' / 'pipeline' / 'card_analysis.py').exists(),
        'RAW_DIR existe': RAW_DIR.exists(),
        'PROCESSED_DIR existe': PROCESSED_DIR.exists(),
        'feature_schema_version': FEATURE_SCHEMA_VERSION,
    }
)

display(repo_status.to_frame(name='valor'))

if not RAW_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta esperada de datos crudos: {RAW_DIR}')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

raw_files = sorted(
    path for path in RAW_DIR.rglob('*') if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS
)
print(f'Imagenes detectadas en data/raw: {len(raw_files)}')
print(f'Ruta de salida esperada para el dataset: {OUTPUT_BASE.with_suffix(".csv")}')

# 3. Fuente de datos

El dataset parte de imagenes propias almacenadas en `data/raw/`.

La etiqueta inicial de cada muestra no se obtiene de una base externa sino de la estructura de carpetas dentro de `data/raw`. En el script `generar_dataset.py`, la funcion `obtener_label_desde_raw(...)` toma la carpeta padre inmediata como etiqueta base, `normalizar_label_condition(...)` la estandariza y `target_desde_label(...)` la transforma a una variable operacional `target_damaged`.

En este esquema:

- `label_condition` representa la etiqueta derivada de la carpeta (`damaged`, `undamaged` o `unknown`).
- `target_damaged` traduce esa etiqueta a una codificacion supervisada operacional (`1`, `0` o `None`).
- `analysis_status` y `usable_for_condition_model` no provienen de la carpeta, sino de la evaluacion tecnica del pipeline sobre cada imagen.

Esto implica limitaciones metodologicas: el etiquetado inicial depende de la organizacion manual del repositorio y no garantiza una certificacion profesional de grading.

In [ ]:
raw_index = pd.DataFrame(
    [
        {
            'relative_path_from_raw': str(path.relative_to(RAW_DIR)),
            'folder_label_raw': obtener_label_desde_raw(path, RAW_DIR),
            'label_condition': normalizar_label_condition(obtener_label_desde_raw(path, RAW_DIR)),
            'target_damaged': target_desde_label(normalizar_label_condition(obtener_label_desde_raw(path, RAW_DIR))),
        }
        for path in raw_files
    ]
)

print('Primeras rutas detectadas en data/raw:')
display(raw_index.head(10))

print('Distribucion inicial derivada de carpetas:')
display(raw_index['label_condition'].value_counts(dropna=False).rename_axis('label_condition').reset_index(name='count'))

# 4. Vista general del pipeline tecnico

La siguiente celda detecta automaticamente que funciones del pipeline existen en esta copia del repo y genera una descripcion del flujo real con los nombres actuales de modulos y funciones.

In [ ]:
PIPELINE_CANDIDATES = {
    'src.utils.image_utils': ['ensure_bgr_uint8'],
    'src.pipeline.card_analysis': ['analyze_card_image'],
    'src.vision.card_detector': ['detect_card_contour'],
    'src.vision.perspective': ['warp_card_perspective'],
    'src.vision.postwarp_validation': ['validate_rectified_card'],
    'src.features.visual_features': ['extract_visual_features'],
    'src.features.geometry_features': ['extract_geometry_features'],
    'src.features.centering_features': ['extract_centering_features'],
    'src.features.edge_features': ['compute_edge_features'],
    'src.features.corner_features': ['compute_corner_features'],
    'src.features.whitening_surface_features': ['compute_whitening_surface_features'],
    'src.features.capture_assessment': ['classify_capture_quality'],
    'src.scoring.condition_score': [
        'compute_capture_quality_score',
        'compute_preliminary_gradix_score',
        'compute_centering_score',
        'compute_edge_score',
        'compute_corner_score',
        'compute_whitening_surface_score',
        'compute_gradix_condition_stub',
        'compute_gradix_condition_stub_v2',
        'compute_gradix_condition_stub_v3',
        'compute_gradix_condition_stub_v4',
    ],
    'generar_dataset': [
        'aplanar_diccionario',
        'evaluar_estado_analisis',
        'procesar_lote_imagenes',
    ],
}

pipeline_rows = []
pipeline_lookup = {}

for module_name, function_names in PIPELINE_CANDIDATES.items():
    module = importlib.import_module(module_name)
    for function_name in function_names:
        exists = hasattr(module, function_name)
        pipeline_rows.append({'module': module_name, 'function': function_name, 'exists': exists})
        pipeline_lookup[(module_name, function_name)] = exists

pipeline_registry = pd.DataFrame(pipeline_rows)
display(pipeline_registry)

def code_name(module_name: str, function_name: str) -> str:
    return f'`{module_name}.{function_name}(...)`'

stage_lines = [
    '## Flujo tecnico detectado en el repo activo',
    '',
    'El pipeline reutilizado por este notebook sigue este orden:',
    f"- Orquestacion principal: {code_name('src.pipeline.card_analysis', 'analyze_card_image')} encapsula la ejecucion completa por imagen.",
    f"- Carga y normalizacion de imagen: el notebook lee la imagen con OpenCV y el pipeline normaliza internamente con {code_name('src.utils.image_utils', 'ensure_bgr_uint8')}.",
    f"- Deteccion de carta: {code_name('src.vision.card_detector', 'detect_card_contour')} estima contorno, esquinas, metricas y overlays de depuracion.",
    f"- Rectificacion / warp: {code_name('src.vision.perspective', 'warp_card_perspective')} rectifica la carta a una vista estabilizada.",
    f"- Validacion post-warp: {code_name('src.vision.postwarp_validation', 'validate_rectified_card')} evalua aspecto, continuidad de bordes externos y riesgo de recorte.",
    '- Extraccion de features sobre la carta rectificada: ' + ', '.join([
        code_name('src.features.visual_features', 'extract_visual_features'),
        code_name('src.features.geometry_features', 'extract_geometry_features'),
        code_name('src.features.centering_features', 'extract_centering_features'),
        code_name('src.features.edge_features', 'compute_edge_features'),
        code_name('src.features.corner_features', 'compute_corner_features'),
        code_name('src.features.whitening_surface_features', 'compute_whitening_surface_features'),
    ]) + '.',
    '- Calculo de scores y stubs de condicion: ' + ', '.join([
        code_name('src.scoring.condition_score', 'compute_capture_quality_score'),
        code_name('src.scoring.condition_score', 'compute_preliminary_gradix_score'),
        code_name('src.scoring.condition_score', 'compute_centering_score'),
        code_name('src.scoring.condition_score', 'compute_edge_score'),
        code_name('src.scoring.condition_score', 'compute_corner_score'),
        code_name('src.scoring.condition_score', 'compute_whitening_surface_score'),
        code_name('src.scoring.condition_score', 'compute_gradix_condition_stub'),
        code_name('src.scoring.condition_score', 'compute_gradix_condition_stub_v2'),
        code_name('src.scoring.condition_score', 'compute_gradix_condition_stub_v3'),
        code_name('src.scoring.condition_score', 'compute_gradix_condition_stub_v4'),
    ]) + '.',
    f"- Evaluacion del estado de analisis: {code_name('src.features.capture_assessment', 'classify_capture_quality')} aporta una lectura cualitativa de captura y {code_name('generar_dataset', 'evaluar_estado_analisis')} decide `analysis_status` y `usable_for_condition_model`.",
    f"- Aplanado tabular: {code_name('generar_dataset', 'aplanar_diccionario')} transforma bloques anidados en columnas planas y {code_name('generar_dataset', 'procesar_lote_imagenes')} recorre `data/raw` para producir el dataset final.",
]

display(Markdown('\n'.join(stage_lines)))

# 5. Ejemplo unitario sobre una imagen

En esta seccion se toma la primera imagen disponible en `data/raw`, se ejecuta el pipeline tecnico real y se inspeccionan sus bloques principales sin usar la interfaz Streamlit.

In [ ]:
if not raw_files:
    raise RuntimeError('No hay imagenes en data/raw para ejecutar el ejemplo unitario.')

sample_path = raw_files[0]
sample_label_raw = obtener_label_desde_raw(sample_path, RAW_DIR)
sample_label_condition = normalizar_label_condition(sample_label_raw)
sample_target_damaged = target_desde_label(sample_label_condition)

sample_bgr = cv2.imread(str(sample_path))
if sample_bgr is None:
    raise RuntimeError(f'No se pudo leer la imagen de ejemplo: {sample_path}')

analysis_unitario = analyze_card_image(sample_bgr)

print(f'Imagen de ejemplo: {sample_path.relative_to(ROOT)}')
print(f'label_condition derivada de carpeta: {sample_label_condition}')
print(f'target_damaged derivado: {sample_target_damaged}')
print(f'debug_images disponibles: {len(analysis_unitario["debug_images"])}')

In [ ]:
def bgr_to_rgb(image: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def show_bgr_gallery(items, cols: int = 3, figsize=(16, 10)) -> None:
    if not items:
        print('No hay imagenes para mostrar.')
        return
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (title, image) in zip(axes.ravel(), items):
        ax.imshow(bgr_to_rgb(image))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

preferred_debug_keys = [
    'detected_contour',
    'final_quad_overlay',
    'candidate_border_support_overlay',
    'multilayer_overlay',
    'warped_with_bbox',
    'edge_overlay',
    'corner_overlay',
]

gallery_items = [('original', sample_bgr)]
debug_images = analysis_unitario['debug_images']
if 'warped_card' in debug_images:
    gallery_items.append(('warped_card', debug_images['warped_card']))
for key in preferred_debug_keys:
    if key in debug_images and key != 'warped_card':
        gallery_items.append((key, debug_images[key]))

show_bgr_gallery(gallery_items[:8], cols=3, figsize=(16, 11))

In [ ]:
def sanitize_for_display(value):
    if isinstance(value, np.ndarray):
        return {'type': 'ndarray', 'shape': list(value.shape), 'dtype': str(value.dtype)}
    if isinstance(value, dict):
        return {k: sanitize_for_display(v) for k, v in value.items()}
    if isinstance(value, list):
        return [sanitize_for_display(v) for v in value]
    if isinstance(value, tuple):
        return tuple(sanitize_for_display(v) for v in value)
    return value

blocks_to_print = {
    'detection': analysis_unitario['detection'],
    'warp': analysis_unitario['warp'],
    'postwarp_validation': analysis_unitario['postwarp_validation'],
    'features': analysis_unitario['features'],
    'scores': analysis_unitario['scores'],
    'assessment': analysis_unitario['assessment'],
    'debug_images_disponibles': sorted(analysis_unitario['debug_images'].keys()),
}

for block_name, block_value in blocks_to_print.items():
    print(f'\n{block_name.upper()}')
    pprint.pp(sanitize_for_display(block_value), sort_dicts=False, width=120)

# 6. Explicacion de la estructura tabular

El script `generar_dataset.py` no guarda el resultado anidado del pipeline tal como sale de `analyze_card_image(...)`. En su lugar, aplana metricas y bloques usando `aplanar_diccionario(...)`, agrega metadata de la imagen, calcula `analysis_status` con `evaluar_estado_analisis(...)` y produce un registro tabular por imagen.

In [ ]:
deteccion = analysis_unitario['detection']
warp_block = analysis_unitario['warp']
postwarp_block = analysis_unitario['postwarp_validation']
features_block = analysis_unitario['features']
scores_block = analysis_unitario['scores']
assessment_block = analysis_unitario['assessment']

used_fallback = deteccion.get('used_fallback', False)
detection_metrics = deteccion.get('metrics', {})
warp_data = warp_block.get('data') if warp_block.get('computed') else None
postwarp_data = postwarp_block.get('data') if postwarp_block.get('computed') else {}

visuales = features_block.get('visual') or {}
geometry = features_block.get('geometry') or {}
centrado = features_block.get('centering') or {}
bordes = features_block.get('edge') or {}
esquinas = features_block.get('corner') or {}
superficie = features_block.get('whitening_surface') or {}

capture_score = scores_block.get('capture_quality') or {}
preliminary_score = scores_block.get('preliminary') or {}
centering_score = scores_block.get('centering') or {}
edge_score = scores_block.get('edge') or {}
corner_score = scores_block.get('corner') or {}
ws_score = scores_block.get('whitening_surface') or {}
stub_v1 = scores_block.get('condition_stub_v1') or {}
stub_v2 = scores_block.get('condition_stub_v2') or {}
stub_v3 = scores_block.get('condition_stub_v3') or {}
stub_v4 = scores_block.get('condition_stub_v4') or {}
capture_assessment = assessment_block.get('capture_quality') or {}

fila_unitaria = {
    'image_id': sample_path.stem,
    'image_filename': sample_path.name,
    'image_path': str(sample_path),
    'relative_path_from_raw': str(sample_path.relative_to(RAW_DIR)),
    'categoria_carpeta_raw': sample_label_raw,
    'label_condition': sample_label_condition,
    'target_damaged': sample_target_damaged,
    'feature_schema_version': FEATURE_SCHEMA_VERSION,
    'det_success': bool(deteccion.get('success', False)),
    'det_used_fallback': bool(used_fallback),
    'warp_success': bool(warp_data and warp_data.get('warped_image') is not None),
    'postwarp_computed': bool(postwarp_block.get('computed', False)),
    'features_computed': bool(features_block.get('computed', False)),
    'scores_computed': bool(scores_block.get('computed', False)),
}

fila_unitaria.update(aplanar_diccionario(detection_metrics, parent_key='det'))
if warp_data is not None:
    fila_unitaria.update(aplanar_diccionario(warp_data.get('metrics', {}), parent_key='warp'))
postwarp_flat = dict(postwarp_data)
postwarp_flat.pop('postwarp_valid', None)
postwarp_flat.pop('postwarp_score', None)
postwarp_flat.pop('retry_recommended', None)
fila_unitaria.update(aplanar_diccionario(postwarp_flat, parent_key='postwarp'))
fila_unitaria.update(aplanar_diccionario(visuales, parent_key='visual'))
fila_unitaria.update(aplanar_diccionario(geometry, parent_key='geometry'))
fila_unitaria.update(aplanar_diccionario(centrado, parent_key='centrado'))
fila_unitaria.update(aplanar_diccionario(bordes, parent_key='borde'))
fila_unitaria.update(aplanar_diccionario(esquinas, parent_key='esquina'))
fila_unitaria.update(aplanar_diccionario(superficie, parent_key='superficie'))
fila_unitaria.update(aplanar_diccionario(capture_score, parent_key='score_capture'))
fila_unitaria.update(aplanar_diccionario(preliminary_score, parent_key='score_prelim'))
fila_unitaria.update(aplanar_diccionario(centering_score, parent_key='score_centering'))
fila_unitaria.update(aplanar_diccionario(edge_score, parent_key='score_edge'))
fila_unitaria.update(aplanar_diccionario(corner_score, parent_key='score_corner'))
fila_unitaria.update(aplanar_diccionario(ws_score, parent_key='score_ws'))
fila_unitaria.update(aplanar_diccionario(stub_v1, parent_key='score_stub_v1'))
fila_unitaria.update(aplanar_diccionario(stub_v2, parent_key='score_stub_v2'))
fila_unitaria.update(aplanar_diccionario(stub_v3, parent_key='score_stub_v3'))
fila_unitaria.update(aplanar_diccionario(stub_v4, parent_key='score_stub_v4'))
fila_unitaria.update(aplanar_diccionario(capture_assessment, parent_key='capture_assessment'))

izq = centrado.get('left_margin', 0)
der = centrado.get('right_margin', 0)
if (izq + der) > 0:
    fila_unitaria['ratio_centrado_calculado'] = round(izq / (izq + der), 3)

fila_unitaria['det_detection_confidence'] = float(detection_metrics.get('detection_confidence', 0.0))
fila_unitaria['det_weak_detection'] = bool(detection_metrics.get('weak_detection', False))
fila_unitaria['postwarp_valid'] = bool(postwarp_data.get('postwarp_valid', False))
fila_unitaria['postwarp_score'] = float(postwarp_data.get('postwarp_score', 0.0))
fila_unitaria['postwarp_retry_recommended'] = bool(postwarp_data.get('retry_recommended', False))

estado_unitario = evaluar_estado_analisis(
    used_fallback=used_fallback,
    detection_metrics=detection_metrics,
    postwarp=postwarp_data,
    visuales=visuales,
    geometry=geometry,
    bordes=bordes,
    esquinas=esquinas,
)
fila_unitaria.update(estado_unitario)

ejemplo_columnas = [
    'label_condition', 'target_damaged', 'analysis_status', 'usable_for_condition_model',
    'det_detection_confidence', 'postwarp_score', 'visual_blur_score', 'visual_brightness_score',
    'geometry_coverage_ratio', 'borde_edge_score', 'esquina_corner_score_raw',
    'superficie_whitening_surface_score', 'score_prelim_gradix_preliminary_score',
    'score_stub_v4_gradix_condition_stub_v4',
]

display(pd.Series({key: fila_unitaria.get(key) for key in ejemplo_columnas if key in fila_unitaria}, name='valor'))

fila_unitaria_df = pd.DataFrame(sorted(fila_unitaria.items()), columns=['column', 'value'])
print(f'Columnas planas generadas para este ejemplo: {len(fila_unitaria_df)}')
display(fila_unitaria_df.head(60))

# 7. Generacion reproducible del dataset completo

La celda siguiente reutiliza la funcion real `procesar_lote_imagenes(...)` definida en `generar_dataset.py` para recorrer `data/raw` y escribir el dataset completo en `data/processed/dataset_gradix.csv` (y, si es posible, tambien en Parquet y Excel).

In [ ]:
procesar_lote_imagenes(str(RAW_DIR), str(OUTPUT_BASE))

dataset_csv_path = OUTPUT_BASE.with_suffix('.csv')
if not dataset_csv_path.exists():
    raise FileNotFoundError(f'No se genero el CSV esperado: {dataset_csv_path}')

df_dataset = pd.read_csv(dataset_csv_path)
print(f'Dataset generado en: {dataset_csv_path}')
print(f'Tamano del dataset: {df_dataset.shape[0]} filas x {df_dataset.shape[1]} columnas')

# 8. Resumen del dataset generado

A continuacion se resume la estructura del CSV generado y la distribucion de variables clave para analisis y modelado.

In [ ]:
if 'df_dataset' not in globals():
    df_dataset = pd.read_csv(OUTPUT_BASE.with_suffix('.csv'))

print('Shape del dataset:')
print(df_dataset.shape)

print('\nPrimeras columnas:')
print(df_dataset.columns.tolist()[:80])

for column_name in ['label_condition', 'target_damaged', 'analysis_status', 'usable_for_condition_model']:
    if column_name in df_dataset.columns:
        print(f'\nvalue_counts de {column_name}:')
        display(df_dataset[column_name].value_counts(dropna=False).rename_axis(column_name).reset_index(name='count'))

pct_invalid_capture = float((df_dataset['analysis_status'] == 'invalid_capture').mean() * 100) if 'analysis_status' in df_dataset.columns else np.nan
pct_not_invalid_capture = 100.0 - pct_invalid_capture if not np.isnan(pct_invalid_capture) else np.nan
pct_usable = float(df_dataset['usable_for_condition_model'].fillna(False).astype(bool).mean() * 100) if 'usable_for_condition_model' in df_dataset.columns else np.nan
pct_not_usable = 100.0 - pct_usable if not np.isnan(pct_usable) else np.nan

resumen_porcentajes = pd.Series(
    {
        'pct_no_invalid_capture': round(pct_not_invalid_capture, 2) if not np.isnan(pct_not_invalid_capture) else None,
        'pct_invalid_capture': round(pct_invalid_capture, 2) if not np.isnan(pct_invalid_capture) else None,
        'pct_usable_for_condition_model': round(pct_usable, 2) if not np.isnan(pct_usable) else None,
        'pct_no_usable_for_condition_model': round(pct_not_usable, 2) if not np.isnan(pct_not_usable) else None,
    },
    name='porcentaje',
)
display(resumen_porcentajes.to_frame())

columnas_muestra = [
    column_name
    for column_name in [
        'image_filename',
        'relative_path_from_raw',
        'label_condition',
        'target_damaged',
        'analysis_status',
        'usable_for_condition_model',
        'det_detection_confidence',
        'postwarp_score',
        'visual_blur_score',
        'geometry_coverage_ratio',
        'score_stub_v4_gradix_condition_stub_v4',
    ]
    if column_name in df_dataset.columns
]
display(df_dataset[columnas_muestra].head(10))

# 9. Limitaciones metodologicas

Este dataset debe interpretarse con varias limitaciones metodologicas:

- Existe sesgo potencial por fondo, iluminacion, distancia de captura y calidad del enfoque.
- Puede haber multiples fotos de la misma carta, por lo que el dataset no implica independencia perfecta entre filas.
- `usable_for_condition_model` es un filtro tecnico del pipeline y no una verdad absoluta sobre la utilidad academica o comercial de la imagen.
- La variable `damaged` / `undamaged` funciona como target supervisado operacional derivado de la organizacion manual de carpetas; no equivale a una certificacion profesional de grading.
- La app y el notebook cumplen roles distintos: este notebook documenta la reproducibilidad del dataset, mientras que `app.py` funciona como interfaz de inferencia y exploracion.

# 10. Cierre

Este notebook produce una documentacion reproducible del origen de datos, el pipeline tecnico por imagen y la generacion del dataset tabular completo de Gradix.

El siguiente paso metodologico recomendado es continuar con `notebooks/02_modelado_y_evaluacion_gradix.ipynb`.

In [ ]:
if 'df_dataset' not in globals():
    df_dataset = pd.read_csv(OUTPUT_BASE.with_suffix('.csv'))

feature_schema_value = None
if 'feature_schema_version' in df_dataset.columns:
    versions = [str(value) for value in df_dataset['feature_schema_version'].dropna().unique().tolist()]
    if len(versions) == 1:
        feature_schema_value = versions[0]
    elif versions:
        feature_schema_value = versions

dataset_report = {
    'total_rows': int(len(df_dataset)),
    'total_columns': int(len(df_dataset.columns)),
    'label_distribution': df_dataset['label_condition'].value_counts(dropna=False).to_dict() if 'label_condition' in df_dataset.columns else {},
    'analysis_status_distribution': df_dataset['analysis_status'].value_counts(dropna=False).to_dict() if 'analysis_status' in df_dataset.columns else {},
    'usable_for_condition_model_distribution': df_dataset['usable_for_condition_model'].value_counts(dropna=False).to_dict() if 'usable_for_condition_model' in df_dataset.columns else {},
    'feature_schema_version': feature_schema_value,
}

dataset_report